### Práctica de transformer

  1. Entrenar un transformer con la función seno y predecir 50 valores subsiguientes.
  2. Entrenar un transformer con el precio diario de Apple (AAPL) y predecir 50 valores subsiguientes.


##### Crea los datos de la función seno

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

# 1. Crear la onda seno
t = np.linspace(0, 100, 1000) # 1000 puntos de 0 a 100
data = np.sin(t).reshape(-1, 1)

# 2. Función para crear ventanas (Secuencias)
def create_sequences(data, seq_length):
    xs, ys = [], []
    for i in range(len(data) - seq_length):
        x = data[i:(i + seq_length)]
        y = data[i + seq_length]
        xs.append(x)
        ys.append(y)
    return np.array(xs), np.array(ys)

seq_length = 50 
X, y = create_sequences(data, seq_length)

# Convertir a Tensores
#device = torch.device('cuda' if torch.cuda.is_available() else 'cpu') # para nvidia
device = torch.device('mps' if torch.mps.is_available() else 'cpu') # para macos
print(device)
X = torch.from_numpy(X).float().to(device)
y = torch.from_numpy(y).float().to(device)

mps


##### Define la red completa

In [ ]:
class SequenceTransformer(nn.Module):
    def __init__(self, d_model=32, nhead=4):
        super(SequenceTransformer, self).__init__()
        # TO-DO

    def forward(self, x): # modelo funcional
        # TO-DO

model = SequenceTransformer().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()

model

##### Entrena

In [ ]:
# TO-DO

##### Calcula los puntos siguientes

In [ ]:
model.eval()
future_points = 200
predictions = []

# Tomamos la última ventana de los datos reales para empezar
last_sequence = X[-1].unsqueeze(0) 

for _ in range(future_points):
    with torch.no_grad():
        pred = model(last_sequence) # Predice el siguiente punto
        predictions.append(pred.item())
        
        # Actualizamos la ventana: desplazamos y añadimos la predicción
        new_row = pred.unsqueeze(1) 
        last_sequence = torch.cat((last_sequence[:, 1:, :], new_row), dim=1)

# Pintar resultados
plt.figure(figsize=(12,5))
plt.plot(t[-300:], data[-300:], label="Datos Reales (Pasado)", color="blue")
plt.plot(np.linspace(t[-1], t[-1]+20, future_points), predictions, label="Predicción Transformer (Futuro)", linestyle="--", color="red")
plt.axvline(x=t[-1], color='black', alpha=0.3)
plt.legend()
plt.title("Generación de Función Seno con Transformer")
plt.show()

#### Acciones de Apple

In [ ]:
%pip install --upgrade yfinance

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt

# Configuración de dispositivo (GPU si está disponible)
device = torch.device('mps' if torch.mps.is_available() else 'cpu')

In [7]:
# Descargar datos
df = yf.download('AAPL', start='2020-01-01', end='2026-01-01')
data = df['Close'].values.reshape(-1, 1)

# Normalización (Importante para Transformers)
scaler = MinMaxScaler(feature_range=(0, 1))
data_scaled = scaler.fit_transform(data)

# Función para crear secuencias
def create_sequences(data, seq_length):
    xs, ys = [], []
    for i in range(len(data) - seq_length):
        x = data[i:(i + seq_length)]
        y = data[i + seq_length]
        xs.append(x)
        ys.append(y)
    return np.array(xs), np.array(ys)

seq_length = 60 # Usamos 60 días previos para predecir el siguiente
X, y = create_sequences(data_scaled, seq_length)

# Convertir a Tensores de PyTorch
X = torch.from_numpy(X).float().to(device)
y = torch.from_numpy(y).float().to(device)

print(f"Forma de X: {X.shape}") # [muestras, seq_length, 1]

[*********************100%***********************]  1 of 1 completed

Forma de X: torch.Size([1448, 60, 1])


In [ ]:
class TimeSeriesTransformer(nn.Module):
    def __init__(self, input_size=1, d_model=64, nhead=4, num_layers=2, dropout=0.1):
        super(TimeSeriesTransformer, self).__init__()
        # TO-DO

    def forward(self, x):
        # TO-DO

model = TimeSeriesTransformer().to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
# entrena
# TO-DO

In [23]:
model.eval()
future_predictions = []
current_batch = X[-1].unsqueeze(0) # Tomamos la última secuencia conocida

for _ in range(50):
    with torch.no_grad():
        pred = model(current_batch)
        future_predictions.append(pred.item())
        
        # Actualizar la ventana: quitar el primero, añadir la predicción al final
        new_pred = pred.unsqueeze(1) # [1, 1, 1]
        current_batch = torch.cat((current_batch[:, 1:, :], new_pred), dim=1)

# Des-normalizar los resultados
future_predictions = scaler.inverse_transform(np.array(future_predictions).reshape(-1, 1))

In [ ]:
plt.figure(figsize=(12,6))
plt.plot(range(len(data)), scaler.inverse_transform(data_scaled), label='Histórico Real')
plt.plot(range(len(data), len(data) + 50), future_predictions, label='Predicción Futura (50 días)', color='red')
plt.title('Predicción AAPL con Transformer')
plt.legend()
plt.show()